# Aula — Caderno 0, Capítulo 1: O ambiente de trabalho

**Livro-curso:** Econofísica Aplicada ao Mercado Financeiro\
**Autor:** Cristiano Alves\
**Duração:** 1h30, com os slides `aula_c0_cap01_ambiente_de_trabalho.pptx`

### Objetivos da aula

1. Explicar por que Python é a ferramenta do curso.
2. Criar um ambiente virtual com Python 3.12 e instalar as bibliotecas em versões fixadas.
3. Trabalhar no JupyterLab e conhecer o Google Colab como alternativa.
4. Organizar uma pasta de projeto em que qualquer pessoa reproduza os seus resultados.

### Como usar este notebook

- Coloque este arquivo na **pasta do projeto**, ao lado do `requirements.txt` (no repositório
  do curso, a pasta `caderno-0/codigo/`).
- Execute as células **na ordem**, com `Shift+Enter`.
- Ao terminar, use *Kernel → Restart Kernel and Run All Cells* e confira se tudo roda do
  início ao fim.

## 1. Por que Python na Econofísica

Séries de milhares de pregões, milhares de simulações, matrizes com centenas de ativos:
nada disso cabe numa planilha. O Python reúne três vantagens para esse trabalho:

| Razão | O que significa |
|---|---|
| **Ecossistema científico** | NumPy, pandas, SciPy e Matplotlib: gratuitos, de código aberto e usados em pesquisa |
| **Legibilidade** | Um código bem escrito se lê quase como a descrição do cálculo, o que facilita conferi-lo |
| **Reprodutibilidade** | Com as versões registradas, o mesmo código produz o mesmo resultado em qualquer computador |

> **Contexto de mercado.** Essas bibliotecas voltam em todo o curso: as caudas pesadas do
> Módulo 2 são medidas com NumPy e SciPy, o expoente de Hurst do Módulo 5 é calculado sobre
> séries do pandas, e as redes de correlação do Módulo 6 nascem de matrizes do NumPy.

## 2. Ambiente virtual e versões fixadas

Um **ambiente virtual** é uma pasta com uma cópia isolada do Python e das bibliotecas de um
projeto. Instalar num ambiente não afeta nenhum outro projeto do computador.

Os comandos abaixo rodam no **terminal**, e não no notebook.

**Linux e macOS:**

```bash
python3.12 -m venv .venv
source .venv/bin/activate
python --version
```

**Windows (Prompt de Comando):**

```bat
py -3.12 -m venv .venv
.venv\Scripts\activate
python --version
```

**Com o ambiente ativado**, instale as versões fixadas e abra o JupyterLab:

```bash
pip install -r requirements.txt
jupyter lab
```

O arquivo `requirements.txt` do Caderno 0:

```text
# Caderno 0: versões fixadas, testadas em conjunto em 10/10/2026
# Python 3.12 ou superior
numpy==2.5.3
pandas==3.0.6
scipy==1.18.1
matplotlib==3.11.2
yfinance==1.7.0
python-bcb==0.4.0
jupyterlab==4.6.4
```

> **Boa prática.** Fixe as versões com `==`, e não com `>=`. Uma versão nova pode mudar uma
> saída, renomear uma função ou alterar um valor padrão.

### 2.1 Em que Python este notebook está rodando?

Num ambiente virtual, `sys.prefix` (o Python em uso) é diferente de `sys.base_prefix` (o
Python a partir do qual o ambiente foi criado).

In [ ]:
import sys

ambiente_virtual = sys.prefix != sys.base_prefix
print("Executável:            ", sys.executable)
print("Versão do Python:      ", sys.version.split()[0])
print("Ambiente virtual ativo:", ambiente_virtual)

**Saída esperada:** o executável dentro da pasta `.venv` do projeto, a versão 3.12 (ou
superior) e `Ambiente virtual ativo: True`. No Google Colab, o resultado é `False`, e isso
é normal (Seção 4).

> **Armadilha comum.** Instalar bibliotecas sem ativar o ambiente: o `pip` instala no Python
> do sistema, e o projeto passa a depender de uma configuração que só existe no seu
> computador. Antes de qualquer `pip install`, procure o `(.venv)` no terminal.

### 2.2 Verificar as versões (Listagem 1.5 do caderno)

In [ ]:
import sys
from importlib.metadata import version

print("Python     ", sys.version.split()[0])
for pacote in ["numpy", "pandas", "scipy", "matplotlib", "yfinance",
               "python-bcb"]:
    print(f"{pacote:<11}", version(pacote))

**Saída esperada:**

```text
Python      3.12.3
numpy       2.5.3
pandas      3.0.6
scipy       1.18.1
matplotlib  3.11.2
yfinance    1.7.0
python-bcb  0.4.0
```

Se a célula parar com `PackageNotFoundError`, alguma biblioteca não está instalada **neste**
Python: o ambiente não foi ativado antes da instalação, ou o notebook está usando outro
kernel. A conferência da próxima seção mostra quais bibliotecas faltam.

### 2.3 Conferência automática contra o `requirements.txt`

Em vez de comparar as versões a olho, o código abaixo lê o `requirements.txt` e confere cada
biblioteca.

In [ ]:
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path


def localizar_requisitos():
    """Procura o requirements.txt na pasta do projeto ou na estrutura do curso."""
    candidatos = ["requirements.txt", "codigo/requirements.txt",
                  "../../caderno-0/codigo/requirements.txt"]
    for candidato in candidatos:
        if Path(candidato).is_file():
            return Path(candidato)
    raise FileNotFoundError("requirements.txt não encontrado: coloque este "
                            "notebook na pasta do projeto")


def ler_requisitos(caminho):
    """Lê as linhas pacote==versão de um requirements.txt."""
    requisitos = {}
    for linha in Path(caminho).read_text(encoding="utf-8").splitlines():
        linha = linha.split("#")[0].strip()
        if "==" in linha:
            pacote, versao = linha.split("==")
            requisitos[pacote.strip()] = versao.strip()
    return requisitos


def conferir_versoes(requisitos):
    """Compara as versões instaladas com as fixadas; True se todas batem."""
    tudo_certo = True
    for pacote, esperada in requisitos.items():
        try:
            instalada = version(pacote)
        except PackageNotFoundError:
            instalada = "ausente"
        confere = instalada == esperada
        tudo_certo = tudo_certo and confere
        situacao = "ok" if confere else "DIFERENTE"
        print(f"{situacao:<10} {pacote:<11} fixada {esperada:<8} instalada {instalada}")
    return tudo_certo

In [ ]:
requisitos = ler_requisitos(localizar_requisitos())
versoes_ok = conferir_versoes(requisitos)
print()
print("Todas as versões conferem." if versoes_ok
      else "Há diferenças: ative o ambiente e refaça a instalação.")

**Saída esperada:** `ok` em todas as linhas e a mensagem *Todas as versões conferem.*

## 3. JupyterLab e notebooks

Um notebook alterna **células de texto** (como esta, em Markdown) e **células de código**.
Cada célula de código é executada com `Shift+Enter`, e a saída aparece logo abaixo dela.

Atalhos úteis no modo de comando (tecle `Esc` para entrar nele):

| Tecla | Ação |
|---|---|
| `A` / `B` | Nova célula acima / abaixo |
| `M` / `Y` | Transformar em célula de texto / de código |
| `D`, `D` | Apagar a célula |
| `Enter` | Voltar a editar a célula |

O exemplo do slide 11, que abre o Capítulo 2:

In [ ]:
preco = 36.77         # R$ por ação: fechamento da PETR4 em 02/01/2025
quantidade = 100
preco * quantidade

**Saída esperada:** `3677.0000000000005`. O motivo desse resto estranho é o tema da primeira
aula do Capítulo 2.

### 3.1 O kernel guarda o que já foi executado

Execute a primeira célula abaixo uma vez. Depois, execute a segunda **várias vezes** e
observe o valor.

In [ ]:
contador = 0

In [ ]:
contador = contador + 1
contador

**Saída esperada:** `1` na primeira execução, `2` na segunda, e assim por diante. O kernel
lembra do que foi executado, e não da ordem em que as células aparecem na página.

> **Armadilha comum.** Executar células fora de ordem. Um notebook que só funciona porque a
> célula 7 rodou antes da célula 3 vai falhar na mão de outra pessoa. Antes de entregar, use
> *Kernel → Restart Kernel and Run All Cells*.

## 4. Google Colab como alternativa

| | JupyterLab local | Google Colab |
|---|---|---|
| **Instalação** | Python e bibliotecas no seu computador | Nada a instalar: roda no navegador |
| **Versões** | Fixadas por você no `requirements.txt` | Escolhidas pelo Google; mudam com o tempo |
| **Arquivos** | Ficam na pasta do projeto | Apagados ao fim da sessão, se não forem salvos no Drive |

No Colab, a primeira célula de cada notebook do curso é esta (Listagem 1.6 do caderno):

```text
!python --version
!pip install -q numpy==2.5.3 pandas==3.0.6 scipy==1.18.1 \
    matplotlib==3.11.2 yfinance==1.7.0 python-bcb==0.4.0
```

A célula abaixo descobre se o notebook está no Colab e, só nesse caso, instala as versões
fixadas. No seu computador, ela não faz nada.

In [ ]:
import subprocess

no_colab = "google.colab" in sys.modules
print("Rodando no Google Colab?", no_colab)

if no_colab:
    pacotes = ["numpy==2.5.3", "pandas==3.0.6", "scipy==1.18.1",
               "matplotlib==3.11.2", "yfinance==1.7.0", "python-bcb==0.4.0"]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pacotes],
                   check=True)
    print("Versões fixadas instaladas. Reinicie a sessão do Colab antes de seguir.")

**Saída esperada no seu computador:** `Rodando no Google Colab? False`.

## 5. Um projeto reprodutível

**Reprodutível** é o projeto em que outra pessoa, com o mesmo código e os mesmos dados,
obtém os mesmos números. Três hábitos garantem isso desde o primeiro dia:

1. **Uma pasta por projeto**, sempre com a mesma estrutura.
2. **Dados brutos nunca são editados**: a limpeza é feita por código, e o resultado vai para
   um arquivo novo.
3. **Sementes fixas** em toda simulação.

A célula abaixo cria as pastas `dados/` e `figuras/` do projeto, se ainda não existirem, e
mostra o conteúdo da pasta.

In [ ]:
def criar_estrutura(raiz="."):
    """Cria as pastas padrão do projeto (não apaga nada que já exista)."""
    for pasta in ["dados", "figuras"]:
        (Path(raiz) / pasta).mkdir(exist_ok=True)


def mostrar_pasta(raiz="."):
    """Lista o primeiro nível da pasta do projeto."""
    raiz = Path(raiz).resolve()
    print(raiz.name + "/")
    for item in sorted(raiz.iterdir()):
        if not item.name.startswith("."):
            print("|--", item.name + ("/" if item.is_dir() else ""))


criar_estrutura()
mostrar_pasta()

### 5.1 Sementes fixas: uma prévia do Capítulo 3

Com a mesma semente, o gerador de números aleatórios produz os mesmos números.

In [ ]:
import numpy as np

rng = np.random.default_rng(seed=42)
print(rng.normal(0, 0.01, size=3).round(5))

rng = np.random.default_rng(seed=42)       # mesma semente...
print(rng.normal(0, 0.01, size=3).round(5))  # ...mesmos números

**Saída esperada:** duas linhas iguais, `[ 0.00305 -0.0104   0.0075 ]`. Em qualquer
computador, com as mesmas versões.

## 6. Prática guiada (15 min)

- [ ] Criar a pasta do projeto e, dentro dela, o ambiente `.venv`
- [ ] Ativar o ambiente e conferir o `(.venv)` no terminal
- [ ] Instalar com `pip install -r requirements.txt`
- [ ] Executar a verificação das versões (Seções 2.2 e 2.3)
- [ ] Criar um notebook com uma célula de texto e uma de código
- [ ] Reiniciar o kernel e rodar todas as células

A célula abaixo faz a conferência final do ambiente.

In [ ]:
problemas = []
if sys.version_info < (3, 12):
    problemas.append(f"Python {sys.version.split()[0]}: o curso exige 3.12 ou superior")
if not (sys.prefix != sys.base_prefix or "google.colab" in sys.modules):
    problemas.append("o notebook não está rodando num ambiente virtual")
if not conferir_versoes(ler_requisitos(localizar_requisitos())):
    problemas.append("há bibliotecas com versão diferente do requirements.txt")
for pasta in ["dados", "figuras"]:
    if not Path(pasta).is_dir():
        problemas.append(f"falta a pasta {pasta}/ (rode a célula da Seção 5)")

print()
if problemas:
    print("Ajustes necessários:")
    for problema in problemas:
        print("  -", problema)
else:
    print("Ambiente pronto para o curso.")

**Saída esperada:** a tabela de versões, toda com `ok`, e a mensagem *Ambiente pronto para o
curso.*

## 7. Exercícios

Os exercícios do Capítulo 1 do caderno.

### Essencial

**Exercício 1.1.** Crie o ambiente virtual do caderno, instale as bibliotecas listadas no
`requirements.txt` e execute a verificação das versões (Seção 2.2). As versões impressas
coincidem com as do livro?

**Exercício 1.2.** Escreva, abaixo, uma célula de texto com o seu nome e uma célula de código
que calcule `2 ** 10`. Depois, reinicie o kernel e execute todas as células.

*Escreva aqui o seu nome (dê dois cliques nesta célula para editá-la).*

In [ ]:
# Exercício 1.2: calcule 2 elevado a 10

### Aplicado

**Exercício 1.3.** Crie, fora da pasta do caderno, a pasta de um projeto seu com a estrutura
da Seção 5, um ambiente virtual próprio e um `requirements.txt` com apenas `numpy` e `pandas`
nas versões deste caderno. Instale e verifique.

**Exercício 1.4.** Use as duas células abaixo. Crie uma variável na primeira, use-a na
segunda e depois **apague a primeira célula**. A segunda continua funcionando? E depois de
reiniciar o kernel? Explique.

In [ ]:
# Exercício 1.4, célula 1: crie aqui uma variável, por exemplo taxa = 0.1375

In [ ]:
# Exercício 1.4, célula 2: use aqui a variável criada na célula 1

### Desafio

**Exercício 1.5.** Repita o Exercício 1.1 no Google Colab, usando a célula da Seção 4. Qual é
a versão do Python no Colab hoje? Que bibliotecas o Colab já traz instaladas, e em que
versões? Discuta o que isso significa para a reprodutibilidade.

## Resumo da aula

- Python, com NumPy, pandas, SciPy e Matplotlib, é a ferramenta de todos os módulos.
- O ambiente virtual isola as bibliotecas; o `requirements.txt` fixa as versões.
- O kernel guarda o que já foi executado: a ordem das células importa.
- O Colab dispensa instalação, mas não controla versões nem guarda arquivos.
- Reprodutibilidade: uma pasta por projeto, dados brutos intocados e sementes fixas.

**Próxima aula:** Capítulo 2 — Python essencial (variáveis, listas, dicionários, funções e
erros), com os preços reais da PETR4.